Why Bidirectional RNN -

Till now what we have been doing is predicting future output with past inputs
Now how to predict past outputs with future inputs 

for e.g ->  in case of NER
![lec_img/rnn_bidirectional.png](lec_img/rnn_bidirectional.png)


Here to remove ambiguity one would have to understand the future context ... and that is where the need for bidirectional rnn comes into play

Also applicable in Machine Translation

![lec_img/rnn_bidirectional1.png](lec_img/rnn_bidirectional1.png)

Here we will have forward as well as a backward RNN working together
The blue one gives the output for forward RNN , the green one for the backward RNN.
Together the yellow shade shows how it understands that the word amazon is a website and not a location from the future ref.

The output from the hidden state is shown for both forward as well as backward propogation...
The thing to note is that for the backward RNN it has initial inputs same as the initial inputs of the forward RNN that is zero or some random initialization.

The output from both the hidden state (ht) will be concatenated for output yt at each timestamp.
v, w, u ->weights in the equations

In [1]:
import tensorflow as tf
from tensorflow.keras.datasets import imdb
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, SimpleRNN, Bidirectional, LSTM, GRU, Dense

In [2]:
num_words = 10000
(X_train, y_train), (X_test, y_test) = imdb.load_data(num_words = num_words)

maxlen = 100
X_train = pad_sequences(X_train, maxlen=maxlen, padding='post', truncating='post')
X_test = pad_sequences(X_test, maxlen=maxlen, padding='post', truncating='post')

embedding_dim = 32

In [3]:
#Simple RNN implementation
model = Sequential([
    Embedding(input_dim=num_words, output_dim=embedding_dim, input_length=maxlen),
    SimpleRNN(5),
    Dense(1, activation='sigmoid')
])
model.build(input_shape=(None, maxlen))
model.summary()

/Users/praveenk/Documents/Deep_learning/tf-metal/lib/python3.12/site-packages/keras/src/layers/core/embedding.py:97: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(
2025-09-19 13:26:18.012062: I metal_plugin/src/device/metal_device.cc:1154] Metal device set to: Apple M3
2025-09-19 13:26:18.012082: I metal_plugin/src/device/metal_device.cc:296] systemMemory: 16.00 GB
2025-09-19 13:26:18.012085: I metal_plugin/src/device/metal_device.cc:313] maxCacheSize: 5.33 GB
2025-09-19 13:26:18.012098: I tensorflow/core/common_runtime/pluggable_device/pluggable_device_factory.cc:305] Could not identify NUMA node of platform GPU ID 0, defaulting to 0. Your kernel may not have been built with NUMA support.
2025-09-19 13:26:18.012105: I tensorflow/core/common_runtime/pluggable_device/pluggable_device_factory.cc:271] Created TensorFlow device (/job:localhost/replica:0/task:0/device:GPU:0 with 0 MB memory) -> physical PluggableDevice (device: 0, name: METAL, pci bus i

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ (None, 100, 32)        │       320,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ simple_rnn (SimpleRNN)          │ (None, 5)              │           190 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 1)              │             6 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 320,196 (1.22 MB)

 Trainable params: 320,196 (1.22 MB)

 Non-trainable params: 0 (0.00 B)

In [4]:
# now lets do Bidirectional RNN
model = Sequential([
    Embedding(input_dim=num_words, output_dim=embedding_dim, input_length=maxlen),
    Bidirectional(SimpleRNN(5)),             # the simplernn is pased through a bidirectional function of keras
    Dense(1, activation='sigmoid')
])
model.build(input_shape=(None, maxlen))
model.summary()

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_1 (Embedding)         │ (None, 100, 32)        │       320,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional (Bidirectional)   │ (None, 10)             │           380 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │            11 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 320,391 (1.22 MB)

 Trainable params: 320,391 (1.22 MB)

 Non-trainable params: 0 (0.00 B)

In [5]:
# the same bidirectional works for LSTM and GRU's
model = Sequential([
    Embedding(input_dim=num_words, output_dim=embedding_dim, input_length=maxlen),
    Bidirectional(LSTM(5)),
    Dense(1, activation='sigmoid')
])
model.build(input_shape=(None, maxlen))
model.summary()

Model: "sequential_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_2 (Embedding)         │ (None, 100, 32)        │       320,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional_1 (Bidirectional) │ (None, 10)             │         1,520 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 1)              │            11 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 321,531 (1.23 MB)

 Trainable params: 321,531 (1.23 MB)

 Non-trainable params: 0 (0.00 B)

In [6]:
model = Sequential([
    Embedding(input_dim=num_words, output_dim=embedding_dim, input_length=maxlen),
    Bidirectional(GRU(5)),
    Dense(1, activation='sigmoid')
])
model.build(input_shape=(None, maxlen))
model.summary()

Model: "sequential_3"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_3 (Embedding)         │ (None, 100, 32)        │       320,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional_2 (Bidirectional) │ (None, 10)             │         1,170 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 1)              │            11 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 321,181 (1.23 MB)

 Trainable params: 321,181 (1.23 MB)

 Non-trainable params: 0 (0.00 B)

Application of this bidirecitonal could be seen in ->

1. NER
2. POS tagging
3. Sentiment Analysis
4. Machine Translation
5. Time series forcasting -> could be helpful in stock price prediction

Drawbacks ->

1. Complexity increases -> as you can see the weights & biases double in the case of bidirecional hidden layers
2. Live speech recognition -> you can't have the data for backward RNN during live speech ... so the model have to wait till the complition of the speech leading to latency... (slow)
